# PINN Digital Twin: Animated Vibration Diagnostics Dashboard

This notebook provides an animated visualization of the PINN-based digital twin for vibration analysis.

**Interactive Mode**: Use the sliders below to adjust physical parameters and see how they affect the system dynamics.

## Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import os
from ipywidgets import interact, FloatSlider, IntSlider, Button, Output, VBox, HBox, Checkbox
from IPython.display import display, clear_output

## 1. Interactive Physical Parameters

Adjust the sliders below to modify the physical parameters of the system.

In [ ]:
# Define sliders for physical parameters
m_slider = FloatSlider(value=10.0, min=1.0, max=50.0, step=0.5, description='Mass [kg]:', style={'description_width': 'initial'})
k_slider = FloatSlider(value=200.0, min=50.0, max=1000.0, step=10.0, description='Stiffness [N/m]:', style={'description_width': 'initial'})
c_slider = FloatSlider(value=15.0, min=1.0, max=100.0, step=1.0, description='Damping [Ns/m]:', style={'description_width': 'initial'})
g_slider = FloatSlider(value=9.81, min=0.0, max=20.0, step=0.1, description='Gravity [m/s²]:', style={'description_width': 'initial'})
x0_slider = FloatSlider(value=1.2, min=-5.0, max=5.0, step=0.1, description='Initial Position [m]:', style={'description_width': 'initial'})
v0_slider = FloatSlider(value=3.0, min=-10.0, max=10.0, step=0.1, description='Initial Velocity [m/s]:', style={'description_width': 'initial'})
t_max_slider = FloatSlider(value=5.0, min=1.0, max=20.0, step=0.5, description='Simulation Time [s]:', style={'description_width': 'initial'})
num_frames_slider = IntSlider(value=250, min=50, max=500, step=25, description='Animation Frames:', style={'description_width': 'initial'})

# Display sliders
print("="*60)
print("PHYSICAL PARAMETERS - Adjust sliders below:")
print("="*60)
display(m_slider)
display(k_slider)
display(c_slider)
display(g_slider)
display(x0_slider)
display(v0_slider)
display(t_max_slider)
display(num_frames_slider)

## 2. PINN Architecture

In [ ]:
class VibrationPINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 40), nn.Tanh(),
            nn.Linear(40, 40), nn.Tanh(),
            nn.Linear(40, 40), nn.Tanh(),
            nn.Linear(40, 3)  # Outputs: [x, y, lambda]
        )
    def forward(self, t): return self.net(t)

## 3. Training and Animation Functions

In [ ]:
def get_physics_loss(model, t_collocation, m, k, c, g):
    """Physics loss function with parameterized physical constants."""
    t_collocation.requires_grad = True
    pred = model(t_collocation)
    x, y, lam = pred[:, 0:1], pred[:, 1:2], pred[:, 2:3]
    dx_dt = torch.autograd.grad(x, t_collocation, torch.ones_like(x), create_graph=True)[0]
    d2x_dt2 = torch.autograd.grad(dx_dt, t_collocation, torch.ones_like(dx_dt), create_graph=True)[0]
    dy_dt = torch.autograd.grad(y, t_collocation, torch.ones_like(y), create_graph=True)[0]
    d2y_dt2 = torch.autograd.grad(dy_dt, t_collocation, torch.ones_like(dy_dt), create_graph=True)[0]

    res_x = m * d2x_dt2 + c * dx_dt + k * x
    res_y = m * d2y_dt2 + lam + m * g
    res_phi = y
    return torch.mean(res_x**2) + torch.mean(res_y**2) + 10 * torch.mean(res_phi**2)

def train_pinn(m, k, c, g, x0, v0, t_max):
    """Train a PINN model with given parameters."""
    pinn = VibrationPINN()
    optimizer = torch.optim.Adam(pinn.parameters(), lr=1e-3)
    t_train = torch.linspace(0, t_max, 600).view(-1, 1)
    
    print(f"\n--- Training PINN ---")
    print(f"Parameters: m={m}kg, k={k}N/m, c={c}Ns/m")
    
    for epoch in range(12001):
        optimizer.zero_grad()
        loss_p = get_physics_loss(pinn, t_train, m, k, c, g)

        t0 = torch.tensor([[0.0]], requires_grad=True)
        p0 = pinn(t0)
        x0_p = p0[:, 0:1]
        v0_p = torch.autograd.grad(x0_p, t0, torch.ones_like(x0_p), create_graph=True)[0]
        loss_ic = (x0_p - x0)**2 + (v0_p - v0)**2 + (p0[:, 1:2])**2

        total_loss = loss_p + 50 * loss_ic
        total_loss.backward()
        optimizer.step()

        if epoch % 3000 == 0:
            print(f"Epoch {epoch:5d} | Loss: {total_loss.item():.8f}")
    
    print("Training complete!")
    return pinn

def get_spring_coords(x_end, num_coils=10, width=0.15):
    """Generate spring coordinates for visualization."""
    x = np.linspace(0, x_end, num_coils * 2 + 1)
    y = np.zeros_like(x)
    y[1:-1:2] = width
    y[2:-1:2] = -width
    return x, y

## 4. Run Dashboard with Current Parameters

In [ ]:
# Output widget for displaying results
dashboard_output = Output()

def run_dashboard(btn=None):
    """
    Run the complete dashboard with current slider values.
    """
    with dashboard_output:
        clear_output(wait=True)
        
        # Get current slider values
        m = m_slider.value
        k = k_slider.value
        c = c_slider.value
        g = g_slider.value
        x0 = x0_slider.value
        v0 = v0_slider.value
        t_max = t_max_slider.value
        num_frames = num_frames_slider.value
        
        # Calculate system properties
        wn = np.sqrt(k/m)
        zeta = c / (2 * np.sqrt(m*k))
        if zeta < 1:
            damping_type = "Underdamped"
            wd = wn * np.sqrt(1 - zeta**2)
        elif zeta == 1:
            damping_type = "Critically Damped"
            wd = 0
        else:
            damping_type = "Overdamped"
            wd = 0
        
        print(f"\n--- System Properties ---")
        print(f"Natural Freq: {wn:.2f} rad/s | Damping Ratio: {zeta:.3f} ({damping_type})")
        
        # Train PINN
        pinn = train_pinn(m, k, c, g, x0, v0, t_max)
        
        # Generate Data via Inference
        t_vals = np.linspace(0, t_max, num_frames)
        t_tensor = torch.tensor(t_vals, dtype=torch.float32, requires_grad=True).view(-1, 1)
        
        # Query the PINN
        pred = pinn(t_tensor)
        x_hist = pred[:, 0].detach().numpy()
        lam_hist = pred[:, 2].detach().numpy()
        
        # Use Automatic Differentiation to get Velocity for the Phase Portrait
        x_tensor = pred[:, 0:1]
        v_tensor = torch.autograd.grad(x_tensor, t_tensor, torch.ones_like(x_tensor))[0]
        v_hist = v_tensor.detach().numpy().flatten()
        
        # Create Dashboard Figure
        fig = plt.figure(figsize=(14, 10))
        fig.suptitle(f"PINN Digital Twin Dashboard\nm={m}kg, k={k}N/m, c={c}Ns/m, ζ={zeta:.3f} ({damping_type})", fontsize=14)

        # Grid Layout
        gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.2])

        # Panel 1: Physical Emulation (Top Left)
        ax_emu = fig.add_subplot(gs[0, 0])
        ax_emu.set_xlim(-0.5, max(x_hist)+1.5)
        ax_emu.set_ylim(-1, 1)
        ax_emu.set_aspect('equal')
        ax_emu.set_title("Mechanical Emulation")
        ax_emu.axvline(0, color='black', lw=4)  # Wall
        mass_rect = plt.Rectangle((0, -0.25), 0.5, 0.5, fc='dodgerblue', ec='black')
        ax_emu.add_patch(mass_rect)
        spring_line, = ax_emu.plot([], [], 'g-', lw=1.5, label='Spring (k)')

        # Damper (Dashpot) Lines
        d_cyl_1, = ax_emu.plot([], [], 'k-', lw=2)
        d_cyl_2, = ax_emu.plot([], [], 'k-', lw=2)
        d_rod, = ax_emu.plot([], [], 'k-', lw=3)
        d_plate, = ax_emu.plot([], [], 'k-', lw=4)

        # Panel 2: Displacement Plot (Top Right)
        ax_x = fig.add_subplot(gs[0, 1])
        ax_x.set_xlim(0, t_max)
        ax_x.set_ylim(min(x_hist)-0.2, max(x_hist)+0.2)
        ax_x.set_title("Displacement x(t)")
        ax_x.set_xlabel("Time [s]")
        ax_x.set_ylabel("Position [m]")
        ax_x.grid(True)
        line_x, = ax_x.plot([], [], 'b-', lw=2)
        point_x, = ax_x.plot([], [], 'ro')

        # Panel 3: Phase Portrait (Bottom Left)
        ax_phase = fig.add_subplot(gs[1, 0])
        ax_phase.set_xlim(min(x_hist)-0.2, max(x_hist)+0.2)
        ax_phase.set_ylim(min(v_hist)-1.0, max(v_hist)+1.0)
        ax_phase.set_title("State-Space Phase Portrait")
        ax_phase.set_xlabel("Displacement x [m]")
        ax_phase.set_ylabel("Velocity v [m/s]")
        ax_phase.grid(True)
        line_phase, = ax_phase.plot([], [], 'purple', lw=2)
        point_phase, = ax_phase.plot([], [], 'ro')

        # Panel 4: Constraint Force (Bottom Right)
        ax_lam = fig.add_subplot(gs[1, 1])
        ax_lam.set_xlim(0, t_max)
        ax_lam.set_ylim(min(lam_hist)-10, max(lam_hist)+10)
        ax_lam.set_title("Identified Constraint Force λ(t)")
        ax_lam.set_xlabel("Time [s]")
        ax_lam.set_ylabel("Normal Force [N]")
        ax_lam.axhline(-m*g, color='red', linestyle='--', label=f'Theoretical -mg = {-m*g:.1f} N')
        ax_lam.grid(True)
        ax_lam.legend()
        line_lam, = ax_lam.plot([], [], 'orange', lw=2)
        point_lam, = ax_lam.plot([], [], 'ro')

        # Animation functions
        def init():
            mass_rect.set_xy((0, -0.25))
            spring_line.set_data([], [])
            for d_line in [d_cyl_1, d_cyl_2, d_rod, d_plate, line_x, point_x, line_phase, point_phase, line_lam, point_lam]:
                d_line.set_data([], [])
            return [mass_rect, spring_line, d_cyl_1, d_cyl_2, d_rod, d_plate, line_x, point_x, line_phase, point_phase, line_lam, point_lam]

        def update(frame):
            t, x, v, lam = t_vals[frame], x_hist[frame], v_hist[frame], lam_hist[frame]

            # 1. Emulation Update
            mass_rect.set_xy((x, -0.25))
            sx, sy = get_spring_coords(x, width=0.1)
            spring_line.set_data(sx, sy + 0.15)

            h, off = 0.1, -0.15
            d_cyl_1.set_data([0, x*0.6], [h+off, h+off])
            d_cyl_2.set_data([0, x*0.6], [-h+off, -h+off])
            d_rod.set_data([x*0.4+0.1, x], [off, off])
            d_plate.set_data([x*0.4+0.1, x*0.4+0.1], [h*0.7+off, -h*0.7+off])

            # 2. Displacement Update
            line_x.set_data(t_vals[:frame], x_hist[:frame])
            point_x.set_data([t], [x])

            # 3. Phase Portrait Update
            line_phase.set_data(x_hist[:frame], v_hist[:frame])
            point_phase.set_data([x], [v])

            # 4. Constraint Force Update
            line_lam.set_data(t_vals[:frame], lam_hist[:frame])
            point_lam.set_data([t], [lam])

            return [mass_rect, spring_line, d_cyl_1, d_cyl_2, d_rod, d_plate, line_x, point_x, line_phase, point_phase, line_lam, point_lam]

        # Create Animation
        ani = FuncAnimation(fig, update, frames=num_frames, init_func=init, blit=True, interval=20)

        plt.tight_layout()
        plt.show()
        
        print(f"\nAnimation complete!")

# Create run button
run_button = Button(description="🚀 Run Dashboard Animation", button_style='success', layout={'width': '250px'})
run_button.on_click(run_dashboard)

print("\n" + "="*60)
print("Click the button below to run the animated dashboard:")
print("="*60)
display(run_button)
display(dashboard_output)

## 5. Quick Static Preview (No Training)

In [ ]:
def quick_preview(m=10.0, k=200.0, c=15.0, g=9.81, x0=1.2, v0=3.0, t_max=5.0):
    """
    Quick preview using numerical solution only.
    Use this to preview system behavior before running the full animation.
    """
    from scipy.integrate import solve_ivp
    
    # Vibration constants
    wn = np.sqrt(k/m)
    zeta = c / (2 * np.sqrt(m*k))
    
    if zeta < 1:
        damping_type = "Underdamped"
        wd = wn * np.sqrt(1 - zeta**2)
    elif zeta == 1:
        damping_type = "Critically Damped"
        wd = 0
    else:
        damping_type = "Overdamped"
        wd = 0
    
    # Numerical solution
    def mbd_ground_truth(t, state):
        x, v = state
        return [v, (-k*x - c*v) / m]
    
    t_eval = np.linspace(0, t_max, 500)
    sol = solve_ivp(mbd_ground_truth, [0, t_max], [x0, v0], t_eval=t_eval, method='RK45')
    
    # Create figure with mechanical emulation and plots
    fig = plt.figure(figsize=(14, 8))
    fig.suptitle(f"Quick Preview: m={m}kg, k={k}N/m, c={c}Ns/m | ζ={zeta:.3f} ({damping_type})", fontsize=12)
    
    # Mechanical Emulation
    ax_emu = fig.add_subplot(2, 2, 1)
    ax_emu.set_xlim(-0.5, max(sol.y[0])+1.5)
    ax_emu.set_ylim(-1, 1)
    ax_emu.set_aspect('equal')
    ax_emu.set_title("Mechanical System (Initial State)")
    ax_emu.axvline(0, color='black', lw=4)
    
    # Draw initial position
    x_init = x0
    mass_rect = plt.Rectangle((x_init, -0.25), 0.5, 0.5, fc='dodgerblue', ec='black')
    ax_emu.add_patch(mass_rect)
    sx, sy = get_spring_coords(x_init, width=0.1)
    ax_emu.plot(sx, sy + 0.15, 'g-', lw=1.5)
    ax_emu.annotate(f'x₀={x0}m', xy=(x_init+0.25, 0.35), fontsize=10, ha='center')
    
    # Displacement plot
    ax_x = fig.add_subplot(2, 2, 2)
    ax_x.plot(t_eval, sol.y[0], 'b-', linewidth=2)
    ax_x.set_title("Displacement Response")
    ax_x.set_xlabel("Time [s]")
    ax_x.set_ylabel("Position x(t) [m]")
    ax_x.grid(True)
    ax_x.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    
    # Phase portrait
    ax_phase = fig.add_subplot(2, 2, 3)
    ax_phase.plot(sol.y[0], sol.y[1], 'r-', linewidth=2)
    ax_phase.plot(sol.y[0][0], sol.y[1][0], 'go', markersize=10, label='Start')
    ax_phase.set_title("Phase Portrait")
    ax_phase.set_xlabel("Displacement [m]")
    ax_phase.set_ylabel("Velocity [m/s]")
    ax_phase.grid(True)
    ax_phase.legend()
    
    # Energy plot
    ax_energy = fig.add_subplot(2, 2, 4)
    KE = 0.5 * m * sol.y[1]**2
    PE = 0.5 * k * sol.y[0]**2
    TE = KE + PE
    ax_energy.plot(t_eval, KE, 'r-', label='Kinetic Energy', alpha=0.7)
    ax_energy.plot(t_eval, PE, 'b-', label='Potential Energy', alpha=0.7)
    ax_energy.plot(t_eval, TE, 'k-', label='Total Energy', linewidth=2)
    ax_energy.set_title("Energy Dissipation")
    ax_energy.set_xlabel("Time [s]")
    ax_energy.set_ylabel("Energy [J]")
    ax_energy.legend()
    ax_energy.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nSystem Properties:")
    print(f"  Natural Frequency (ωn): {wn:.2f} rad/s ({wn/(2*np.pi):.2f} Hz)")
    print(f"  Damping Ratio (ζ): {zeta:.3f} ({damping_type})")
    if zeta < 1:
        print(f"  Damped Frequency (ωd): {wd:.2f} rad/s ({wd/(2*np.pi):.2f} Hz)")
        print(f"  Period: {2*np.pi/wd:.3f} s")
    print(f"  Initial Energy: {TE[0]:.2f} J")
    print(f"  Final Energy: {TE[-1]:.2f} J")
    print(f"  Energy Dissipated: {TE[0]-TE[-1]:.2f} J ({100*(TE[0]-TE[-1])/TE[0]:.1f}%)")

# Create interactive widget
print("\n" + "="*60)
print("QUICK PREVIEW (Numerical Solution Only)")
print("="*60)
interact(quick_preview,
         m=FloatSlider(value=10.0, min=1.0, max=50.0, step=0.5, description='Mass [kg]:'),
         k=FloatSlider(value=200.0, min=50.0, max=1000.0, step=10.0, description='Stiffness [N/m]:'),
         c=FloatSlider(value=15.0, min=1.0, max=100.0, step=1.0, description='Damping [Ns/m]:'),
         g=FloatSlider(value=9.81, min=0.0, max=20.0, step=0.1, description='Gravity [m/s²]:'),
         x0=FloatSlider(value=1.2, min=-5.0, max=5.0, step=0.1, description='Init Pos [m]:'),
         v0=FloatSlider(value=3.0, min=-10.0, max=10.0, step=0.1, description='Init Vel [m/s]:'),
         t_max=FloatSlider(value=5.0, min=1.0, max=20.0, step=0.5, description='Time [s]:'));